# Danish Road Trip 🚗🇩🇰
Visit 30 Danish towns, from Skagen to Gedser-ish, and come home, in the shortest round trip (straight-line distances). It's the travelling salesman problem: $29!/2 \approx 4\cdot10^{30}$ routes. Evolve good routes with a **genetic algorithm** and compare with a classic heuristic (nearest neighbour + 2-opt).

Background: [[Calculus and Optimization]] (optimisation without gradients) · overview: [[Fun Projects]]

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import numpy as np, matplotlib.pyplot as plt
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing

TOWNS = {"København": (55.676, 12.568), "Aarhus": (56.157, 10.211), "Odense": (55.403, 10.402), "Aalborg": (57.048, 9.919), "Esbjerg": (55.476, 8.459),
         "Randers": (56.461, 10.036), "Kolding": (55.491, 9.472), "Horsens": (55.861, 9.850), "Vejle": (55.709, 9.536), "Roskilde": (55.642, 12.080),
         "Herning": (56.139, 8.974), "Silkeborg": (56.170, 9.545), "Næstved": (55.230, 11.760), "Fredericia": (55.566, 9.753), "Viborg": (56.453, 9.402),
         "Køge": (55.458, 12.182), "Holstebro": (56.360, 8.616), "Slagelse": (55.403, 11.354), "Hillerød": (55.927, 12.300), "Sønderborg": (54.914, 9.792),
         "Svendborg": (55.060, 10.607), "Hjørring": (57.464, 9.982), "Frederikshavn": (57.441, 10.537), "Skagen": (57.720, 10.583), "Ribe": (55.328, 8.762),
         "Thisted": (56.957, 8.694), "Nykøbing F": (54.769, 11.874), "Helsingør": (56.036, 12.611), "Grenaa": (56.416, 10.879), "Skive": (56.567, 9.027)}
names = list(TOWNS); ll = np.radians(np.array(list(TOWNS.values())))
dlat = ll[:, None, 0] - ll[None, :, 0]; dlon = ll[:, None, 1] - ll[None, :, 1]
D = 2 * 6371 * np.arcsin(np.sqrt(np.sin(dlat / 2) ** 2 + np.cos(ll[:, None, 0]) * np.cos(ll[None, :, 0]) * np.sin(dlon / 2) ** 2))   # haversine, km
print(f"{len(names)} towns; Aalborg–København {D[names.index('Aalborg'), names.index('København')]:.0f} km as the crow flies")
def plot(route, title):
    xy = np.array([TOWNS[names[i]][::-1] for i in list(route) + [route[0]]])
    plt.figure(figsize=(4.5, 5)); plt.plot(*xy.T, "o-"); plt.title(title)
    for i in route: plt.annotate(names[i], TOWNS[names[i]][::-1], fontsize=7)
    plt.show()

## 1. Length of a round trip
`route` is a permutation of town indices; return the total distance **including the leg back to the start**.

In [ ]:
def route_length(route, D):
    # TODO 1: sum of D between consecutive towns, wrapping around
    raise NotImplementedError  # TODO 1

In [ ]:
check("route length", lambda: route_length([0, 1, 2], np.array([[0, 1, 5], [1, 0, 2], [5, 2, 0.0]])) == 8.0, "0→1 (1) + 1→2 (2) + 2→0 (5) = 8.")

## 2. Order crossover (OX)
Children must still be permutations, so normal crossover doesn't work. **OX:** copy a random slice `p1[i:j+1]` into the child at the same positions, then fill the remaining positions, starting after `j` and wrapping around, with the missing towns **in the order they appear in `p2`**.

In [ ]:
def order_crossover(p1, p2, rng):
    # TODO 2: keep a slice of p1, fill the rest in p2's order (starting after the slice, wrapping)
    raise NotImplementedError  # TODO 2

In [ ]:
def _t_ox():
    rng = np.random.default_rng(3); ok = True
    for _ in range(50):
        p1 = list(rng.permutation(10)); p2 = list(rng.permutation(10)); c = order_crossover(p1, p2, np.random.default_rng(7))
        r = np.random.default_rng(7); i, j = sorted(r.choice(10, 2, replace=False))
        ok &= sorted(c) == list(range(10)) and c[i:j + 1] == p1[i:j + 1]
    return ok
check("order crossover", _t_ox, "The child must be a permutation and contain p1's slice at the same positions (draw i, j with rng.choice(n, 2, replace=False)).")

In [ ]:
def two_opt(route):
    route = list(route); improved = True
    while improved:
        improved = False
        for i in range(1, len(route) - 1):
            for j in range(i + 1, len(route)):
                new = route[:i] + route[i:j + 1][::-1] + route[j + 1:]
                if route_length(new, D) < route_length(route, D) - 1e-9: route, improved = new, True
    return route

def genetic(pop_size=200, gens=400, seed=0):
    rng = np.random.default_rng(seed); n = len(names)
    pop = [list(rng.permutation(n)) for _ in range(pop_size)]; best_hist = []
    for g in range(gens):
        fit = np.array([route_length(r, D) for r in pop]); order = np.argsort(fit); best_hist.append(fit[order[0]])
        elite = [pop[i] for i in order[:20]]
        def pick():                                                      # tournament selection
            a, b = rng.choice(pop_size, 2, replace=False); return pop[a] if fit[a] < fit[b] else pop[b]
        children = []
        while len(children) < pop_size - len(elite):
            c = order_crossover(pick(), pick(), rng)
            if rng.random() < 0.3:                                       # mutation: reverse a segment
                i, j = sorted(rng.choice(n, 2, replace=False)); c[i:j + 1] = c[i:j + 1][::-1]
            children.append(c)
        pop = elite + children
    fit = [route_length(r, D) for r in pop]; return pop[int(np.argmin(fit))], best_hist

if ready("route length", "order crossover"):
    start = 0; left = set(range(1, len(names))); nn = [start]
    while left: nxt = min(left, key=lambda j: D[nn[-1], j]); nn.append(nxt); left.remove(nxt)
    opt = two_opt(nn); ga, hist = genetic()
    L_rand = np.mean([route_length(np.random.default_rng(s).permutation(len(names)), D) for s in range(100)])
    print(f"random route {L_rand:.0f} km | nearest neighbour {route_length(nn, D):.0f} km | NN + 2-opt {route_length(opt, D):.0f} km | genetic algorithm {route_length(ga, D):.0f} km")
    plt.plot(hist); plt.xlabel("generation"); plt.ylabel("best route (km)"); plt.show(); plot(ga, f"GA: {route_length(ga, D):.0f} km")
    check("GA finds a good tour", lambda: route_length(ga, D) < 1.1 * route_length(opt, D) and route_length(ga, D) < 0.4 * L_rand,
          "The GA should get within 10 % of NN+2-opt and below 40 % of a random tour.")

<details><summary>▶ Solution</summary>

```python
def route_length(route, D):
    return float(sum(D[route[i], route[(i + 1) % len(route)]] for i in range(len(route))))
```

```python
def order_crossover(p1, p2, rng):
    n = len(p1)
    i, j = sorted(rng.choice(n, 2, replace=False))
    child = [-1] * n
    child[i:j + 1] = p1[i:j + 1]
    fill = [g for g in p2 if g not in child[i:j + 1]]
    k = 0
    for pos in list(range(j + 1, n)) + list(range(0, i)):
        child[pos] = fill[k]; k += 1
    return child
```
</details>

## Stretch goals
- Real **driving** distances: the Great Belt bridge and ferries change everything. Get a distance matrix from OpenStreetMap routing (OSRM) and rerun.
- Add 2-opt as a mutation operator (a *memetic* algorithm). How much faster does the GA converge?
- Simulated annealing on the same problem: fewer moving parts than a GA.